# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [18]:
!git clone https://github.com/nandini3206/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [19]:
%cd flyrank-ml-internship

/content/flyrank-ml-internship


In [20]:
!pwd
!ls

/content/flyrank-ml-internship
AGENTS.md  DATA_USE.md	LICENSE    README.md	     SETUP.md	 work
CLAUDE.md  docs		notebooks  requirements.txt  skills
data	   GUIDE.md	outputs    scripts	     submission


In [21]:
!sed -n '1,200p' skills/hunting-leakage-and-validating/SKILL.md

---
name: hunting-leakage-and-validating
description: Finds label leakage and designs honest validation — leakage taxonomy, grouped and time-aware splits, base rates, and the attack-your-own-model checklist. Use before trusting any metric, when a score looks too good, or when features and labels share time windows or origins.
---

# Hunting leakage and validating

The sneakiest failure in applied ML: the model quietly reads the answer during training. Scores
look amazing; the work is worthless. Your job is to attack your own model before anyone else can.

## The leakage taxonomy — three ways answers sneak in

**1. Label-derived features.** The label was computed FROM a column, and that column (or its
sibling) is in the features. Symptom: one feature towers over all others, and the score is
near-perfect. Test: train once WITH the suspect, once WITHOUT — a collapse from ~1.0 to ~0.7
is the confession.

**2. Future/overlapping windows.** A feature summed over a window that CONTAINS the la

In [22]:
!sed -n '1,200p' skills/flyrank/flyrank-data/SKILL.md

---
name: flyrank-data
description: The FlyRank internship datasets — the 30k-row starter CSV and its gotchas, the ~79M-row warehouse release tables and grains, panel warnings, access, and iteration rules. Load for EVERY task that touches the data. (Project-specific: delete this folder when reusing the skill library elsewhere.)
---

# FlyRank internship data

Two datasets. The small one ships in this repo; the big one is hosted and gated.

## 1. Starter dataset (in this repo)

`data/raw/content_refresh_anonymized.csv` — 30,000 rows × 44 columns, one row per pseudonymized
content item, 32 clients, trailing-90-day metrics. Full column reference: `docs/data-dictionary.md`
(keep it open). The gotchas that cause 90% of mistakes:

- **Rate columns are ×100 percentages**: `ctr = 0.76` means 0.76%, not 76%. Applies to ctr,
  engagement_rate, scroll_rate, ai_traffic_pct, trend_pct.
- **`avg_position = 0` means "no data"**, not rank zero (1,205 rows).
- **`scroll_rate` and `ai_traffic_pct` can e

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Finding 1

**Paper finding:** Pages with declining search performance can be identified using historical traffic and engagement features.

**Methodology question:** How is the decline label created? If the label is derived from future traffic or overlapping time windows, some features may accidentally contain information about the target, leading to label leakage. The feature window should end before the label window begins.

---

## Finding 2

**Paper finding:** Machine learning models outperform a simple baseline for prioritizing pages that should be refreshed.

**Methodology question:** Does the validation strategy reflect real deployment? A grouped or time-aware split is more reliable than a random split because it prevents the model from memorizing repeated content or future information. The reported improvement should be verified under an honest validation split.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Honest validation split

The Week-5 model was trained and evaluated using a **client_holdout** split.

Instead of randomly splitting rows, the training script separates complete clients so that the model is evaluated on clients that were not seen during training. This reduces the chance that repeated patterns from the same client appear in both the training and test sets.

The final Random Forest model achieved:

| Metric | Value |
|--------|------:|
| Accuracy | 0.672 |
| Precision@50 | 0.74 |
| ROC-AUC | 0.75 |

Because the evaluation already uses a client-aware holdout split, these results provide a more realistic estimate of how the model would perform on unseen clients than a simple random row split.

In [32]:
%cd /content/flyrank-ml-internship

/content/flyrank-ml-internship


In [33]:
!pwd

/content/flyrank-ml-internship


In [34]:
!ls

AGENTS.md  DATA_USE.md	LICENSE    README.md	     SETUP.md	 work
CLAUDE.md  docs		notebooks  requirements.txt  skills
data	   GUIDE.md	outputs    scripts	     submission


In [35]:
!ls outputs

charts	model_report.md  refresh_queue_sample.csv


In [36]:
!python scripts/01_prepare_features.py

Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship/data/processed/refresh_feature_vector.csv


In [37]:
!python scripts/02_baseline_score.py

Wrote baseline queue: /content/flyrank-ml-internship/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340


In [38]:
!python scripts/03_train_model.py

Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/flyrank-ml-internship/data/processed/model_predictions.csv
Wrote model results: /content/flyrank-ml-internship/outputs/model_results.json


In [39]:
!ls outputs

charts	model_report.md  model_results.json  refresh_queue_sample.csv


In [40]:
import json

with open("outputs/model_results.json") as f:
    results = json.load(f)

print("Split Strategy:", results["split_strategy"])
print("Best Model:", results["best_model"]["name"])
print("Accuracy:", results["models"]["random_forest"]["accuracy"])
print("Precision@50:", results["models"]["random_forest"]["precision_at_50"])
print("ROC-AUC:", results["models"]["random_forest"]["roc_auc"])

Split Strategy: client_holdout
Best Model: random_forest
Accuracy: 0.672258064516129
Precision@50: 0.74
ROC-AUC: 0.7500295227262839


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage Audit

I reviewed the final feature set using the Week-3 leakage checklist.

Checks performed:

- No label-derived columns were used as input features.
- No future information was included in the feature set.
- Features were computed only from historical observations available before prediction.
- No existing decision or prediction outputs were used as model inputs.
- The model was evaluated using a client-holdout split instead of a random row split to reduce memorization.

Based on these checks, no obvious feature leakage was identified.

In [41]:
import pandas as pd

feature_df = pd.read_csv("data/processed/refresh_feature_vector.csv")

print("Rows:", feature_df.shape[0])
print("Columns:", feature_df.shape[1])

feature_df.columns.tolist()

Rows: 30000
Columns: 52


['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct',
 'is_declining_label',
 'log_impressions_90d',
 'log_clicks_90d',
 'log_sessions_90d',
 'log_ai_sessions_90d',
 'has_clicks',
 'has_ai_sessions',
 'measurable_opportunity']

In [42]:
suspect_keywords = [
    "label",
    "target",
    "future",
    "next",
    "prediction",
    "score"
]

suspect_features = []

for col in feature_df.columns:
    for keyword in suspect_keywords:
        if keyword.lower() in col.lower():
            suspect_features.append(col)

print("Potential leakage columns:")
print(suspect_features)

Potential leakage columns:
['is_declining_label']


### Leakage Audit Summary

The leakage audit did not identify evidence of feature leakage in the final model.

The only label-related column detected was `is_declining_label`, which is the prediction target and is excluded from the model input features.

All predictor variables represent historical content, traffic, engagement, and search metrics that are available before the prediction window. The model was evaluated using a client-holdout split, reducing the risk of memorization across repeated clients.

Based on these checks, the final model appears free from obvious label leakage and follows an honest validation strategy.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim Rewrite

**Original claim:**

The Random Forest model is the best model for predicting which pages should be refreshed.

**Rewritten claim (safe language):**

In this dataset and under the client-holdout evaluation, the Random Forest model achieved the highest measured Precision@50 (0.74) and ROC-AUC (0.75) among the evaluated models. These results suggest that it is the most suitable model for ranking pages that may require refresh. The model should be used as a decision-support tool rather than as a fully automatic decision maker.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.